# Entrenar "Hey Sokari"

Entrena el detector de **"Hey Sokari"** con cientos de voces sintéticas distintas, para que responda a cualquiera.
Al final se descarga un archivo **`hey_sokari.jww`**.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **T4 GPU** → Guardar.
2. *Entorno de ejecución →* **Ejecutar todo**.

Tarda más o menos 1–2 horas. Si Colab se desconecta, vuelve a darle *Ejecutar todo*: lo que ya se generó no se repite.

**Si algo falla:** copia completo el mensaje de la celda que salió en rojo y mándaselo a Claude.

## 1. Revisar la GPU y la versión de Python

In [ ]:
import subprocess, sys
print("Python", sys.version.split()[0])
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "SIN GPU")
assert gpu, "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU, y dale Ejecutar todo otra vez."
if sys.version_info >= (3, 13):
    print("AVISO: las voces sintéticas (piper-phonemize) solo tienen paquetes hasta Python 3.12; la instalación puede fallar.")

## 2. Instalar openWakeWord y el generador de voces

In [ ]:
%%bash
set -e
cd /content
[ -d openWakeWord ] || git clone -q --depth 1 https://github.com/dscripka/openWakeWord
pip install -q -e ./openWakeWord --no-deps
[ -d piper-sample-generator ] || git clone -q --depth 1 https://github.com/rhasspy/piper-sample-generator
mkdir -p piper-sample-generator/models
MODELO=piper-sample-generator/models/en_US-libritts_r-medium.pt
[ -s $MODELO ] || wget -q -O $MODELO https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
pip install -q piper-phonemize webrtcvad mutagen torchinfo torchmetrics "speechbrain<1" audiomentations \
    torch-audiomentations acoustics pronouncing "datasets<3" deep-phonemizer onnx onnxruntime pyyaml
echo "Instalado."

## 3. Bajar los audios de apoyo

- Respuestas de cuartos reales (eco), ruido de fondo y música: para que funcione en casa, no solo en silencio.
- Rasgos ya calculados de ~2,000 horas de habla y ruido que **no** son "Hey Sokari" (unos 17 GB): con ellos aprende a no activarse con cualquier cosa.

In [ ]:
import os
from pathlib import Path
import numpy as np
import scipy.io.wavfile
import datasets
from tqdm.auto import tqdm
os.chdir("/content")

def guardar(carpeta, nombre, audio):
    scipy.io.wavfile.write(os.path.join(carpeta, nombre), 16000, (np.clip(audio, -1, 1) * 32767).astype(np.int16))

os.makedirs("mit_rirs", exist_ok=True)
if len(os.listdir("mit_rirs")) < 200:
    for row in tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True), desc="ecos"):
        guardar("mit_rirs", row["audio"]["path"].split("/")[-1], row["audio"]["array"])

os.makedirs("audioset_16k", exist_ok=True)
if len(os.listdir("audioset_16k")) < 500:
    if not Path("audioset/audio").exists():
        os.makedirs("audioset", exist_ok=True)
        !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/bal_train09.tar
        !tar -xf audioset/bal_train09.tar -C audioset
    ds = datasets.Dataset.from_dict({"audio": [str(p) for p in Path("audioset/audio").glob("**/*.flac")]})
    for row in tqdm(ds.cast_column("audio", datasets.Audio(sampling_rate=16000)), desc="ruido"):
        guardar("audioset_16k", Path(row["audio"]["path"]).stem + ".wav", row["audio"]["array"])

os.makedirs("fma", exist_ok=True)
if len(os.listdir("fma")) < 100:
    fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
               .cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for _ in tqdm(range(3600 // 30), desc="música"):
        row = next(fma)
        guardar("fma", row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav"), row["audio"]["array"])

for f in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"]:
    if not os.path.exists(f) or os.path.getsize(f) < 1_000_000:
        !wget -q -O {f} https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/{f}
print("Audios listos.")

## 4. Configurar el entrenamiento

`CALIDAD = "buena"` hace ~15,000 ejemplos (más lento, mejor). `"rapida"` sirve para probar que todo corre.

In [ ]:
import yaml
CALIDAD = "buena"
n, pasos = (15000, 40000) if CALIDAD == "buena" else (3000, 10000)
config = {
    "model_name": "hey_sokari",
    "target_phrase": ["hey sokari", "hey so kari"],
    # Frases que suenan parecido y NO deben activarlo (además, openWakeWord arma otras solo).
    # Ninguna demasiado parecida a "sokari": eso le enseñaría a rechazar tu propia pronunciación.
    "custom_negative_phrases": ["hey sakura", "hey safari", "hey sorry", "hey calgary", "hey siri", "hey karen",
                                "hey soccer", "hey sugar", "hey sophie"],
    "n_samples": n,
    "n_samples_val": max(1000, n // 10),
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./modelo",
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./audioset_16k", "./fma"],
    "background_paths_duplication_rate": [1, 1],
    "false_positive_validation_data_path": "validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    # El modelo que Sokari sabe leer: "dnn" con una capa oculta.
    "model_type": "dnn",
    "layer_size": 32,
    "steps": pasos,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
}
with open("hey_sokari.yaml", "w") as f:
    yaml.dump(config, f)
print(open("hey_sokari.yaml").read())

## 5. Generar voces, mezclarlas con ruido y entrenar

In [ ]:
%%bash
cd /content
set -e
python openWakeWord/openwakeword/train.py --training_config hey_sokari.yaml --generate_clips
python openWakeWord/openwakeword/train.py --training_config hey_sokari.yaml --augment_clips
# Al final intenta pasarlo también a tflite; si solo eso falla no importa: Sokari usa el .onnx.
python openWakeWord/openwakeword/train.py --training_config hey_sokari.yaml --train_model || true
test -s modelo/hey_sokari.onnx || { echo "No se generó modelo/hey_sokari.onnx: revisa los mensajes de arriba."; exit 1; }
ls -la modelo/hey_sokari.onnx

## 6. Convertir al formato de Sokari (se revisa solo)

In [ ]:
%%writefile onnx_a_jww.py
#!/usr/bin/env python3
"""Convierte el clasificador que entrena openWakeWord (hey_sokari.onnx) al
formato que Sokari lleva adentro del exe (res/hey_sokari.jww).

Se revisa solo: calcula la salida con los pesos convertidos (la misma cuenta
que hace Sokari en C) y la compara con onnxruntime sobre el modelo original.
Si no coinciden, no escribe nada.

Uso:  python onnx_a_jww.py hey_sokari.onnx hey_sokari.jww
Pide: pip install numpy onnx onnxruntime
"""
import hashlib
import struct
import sys

import numpy as np

FEAT_FRAMES, EMB_DIM = 16, 96  # lo que Sokari le pasa al clasificador
IN_DIM = FEAT_FRAMES * EMB_DIM
ORDER = ["fc1.w", "fc1.b", "ln1.w", "ln1.b", "fc2.w", "fc2.b", "ln2.w", "ln2.b", "fc3.w", "fc3.b"]
# Nombres que les pone PyTorch al exportar el modelo "dnn" de openWakeWord.
TORCH_NAMES = {
    "fc1.w": "layer1.weight", "fc1.b": "layer1.bias",
    "ln1.w": "layernorm1.weight", "ln1.b": "layernorm1.bias",
    "fc2.w": "blocks.0.fcn_layer.weight", "fc2.b": "blocks.0.fcn_layer.bias",
    "ln2.w": "blocks.0.layer_norm.weight", "ln2.b": "blocks.0.layer_norm.bias",
    "fc3.w": "last_layer.weight", "fc3.b": "last_layer.bias",
}


class ConversionError(Exception):
    pass


def load_constants(model):
    from onnx import numpy_helper
    consts = {i.name: numpy_helper.to_array(i) for i in model.graph.initializer}
    for n in model.graph.node:
        if n.op_type == "Constant":
            for a in n.attribute:
                if a.name == "value":
                    consts[n.output[0]] = numpy_helper.to_array(a.t)
    return {k: np.asarray(v, dtype=np.float32) for k, v in consts.items()}


def by_name(consts):
    if all(v in consts for v in TORCH_NAMES.values()):
        return {k: consts[v] for k, v in TORCH_NAMES.items()}
    return None


def by_graph(model, consts):
    """Si los nombres no son los de PyTorch: recorre el grafo en orden y toma
    las 3 capas lineales (Gemm, o MatMul + Add) y las 2 LayerNorm (el operador,
    o la versión desarmada que termina en Div -> Mul(escala) -> Add(sesgo))."""
    nodes = list(model.graph.node)
    users = {}
    for n in nodes:
        for x in n.input:
            users.setdefault(x, []).append(n)

    def next_op(out, op):
        return next((u for u in users.get(out, []) if u.op_type == op), None)

    def const_input(n):
        return next((consts[x] for x in n.input if x in consts), None)

    linears, norms = [], []
    for n in nodes:
        if n.op_type == "Gemm" and n.input[1] in consts:
            w = consts[n.input[1]]
            if not next((a.i for a in n.attribute if a.name == "transB"), 0):
                w = w.T
            b = consts.get(n.input[2]) if len(n.input) > 2 else None
            linears.append((w, b))
        elif n.op_type == "MatMul" and n.input[1] in consts:
            add = next_op(n.output[0], "Add")
            linears.append((consts[n.input[1]].T, const_input(add) if add is not None else None))
        elif n.op_type == "LayerNormalization":
            norms.append((consts.get(n.input[1]), consts.get(n.input[2]) if len(n.input) > 2 else None))
        elif n.op_type == "Div":
            mul = next_op(n.output[0], "Mul")
            add = next_op(mul.output[0], "Add") if mul is not None else None
            if mul is not None and add is not None:
                norms.append((const_input(mul), const_input(add)))
    if len(linears) != 3 or len(norms) != 2:
        return None
    (w1, b1), (w2, b2), (w3, b3) = linears
    (g1, e1), (g2, e2) = norms
    return dict(zip(ORDER, [w1, b1, g1, e1, w2, b2, g2, e2, w3, b3]))


def check_shapes(p):
    if any(p[k] is None for k in ORDER):
        raise ConversionError("al modelo le falta algún peso (¿no es el modelo 'dnn' de openWakeWord?)")
    p = {k: np.ascontiguousarray(np.asarray(v, dtype=np.float32).reshape(v.shape)) for k, v in p.items()}
    for k in ["fc1.b", "ln1.w", "ln1.b", "fc2.b", "ln2.w", "ln2.b", "fc3.b"]:
        p[k] = p[k].reshape(-1)
    if p["fc1.w"].ndim != 2 or p["fc1.w"].shape[1] != IN_DIM:
        raise ConversionError(f"la primera capa recibe {p['fc1.w'].shape}; Sokari le pasa {FEAT_FRAMES}x{EMB_DIM} "
                              f"rasgos ({IN_DIM}). ¿La frase dura más de 2 segundos?")
    h = p["fc1.w"].shape[0]
    if not 1 <= h <= 256:
        raise ConversionError(f"capa oculta de {h} neuronas; Sokari acepta de 1 a 256")
    want = {"fc1.b": (h,), "ln1.w": (h,), "ln1.b": (h,), "fc2.w": (h, h), "fc2.b": (h,), "ln2.w": (h,),
            "ln2.b": (h,), "fc3.w": (1, h), "fc3.b": (1,)}
    for k, shape in want.items():
        if k == "fc3.w":
            p[k] = p[k].reshape(1, -1)
        if p[k].shape != shape:
            raise ConversionError(f"{k} tiene forma {p[k].shape}, se esperaba {shape}")
    return p


def forward(p, feats):
    """La misma cuenta que wakeword.c: fc1 -> LayerNorm -> ReLU -> fc2 ->
    LayerNorm -> ReLU -> fc3 -> sigmoide."""
    def ln_relu(x, g, b):
        m = x.mean(axis=1, keepdims=True)
        v = ((x - m) ** 2).mean(axis=1, keepdims=True)
        return np.maximum((x - m) / np.sqrt(v + 1e-5) * g + b, 0)

    x = feats.reshape(len(feats), -1).astype(np.float64)
    x = ln_relu(x @ p["fc1.w"].T + p["fc1.b"], p["ln1.w"], p["ln1.b"])
    x = ln_relu(x @ p["fc2.w"].T + p["fc2.b"], p["ln2.w"], p["ln2.b"])
    z = x @ p["fc3.w"].T + p["fc3.b"]
    return (1 / (1 + np.exp(-z))).reshape(-1)


def pack(p):
    out = [b"JWW1", struct.pack("<I", len(ORDER))]
    for k in ORDER:
        name = ("kw." + k).encode()
        a = np.ascontiguousarray(p[k], dtype="<f4")
        out += [struct.pack("<I", len(name)), name, struct.pack("<I", a.ndim),
                struct.pack(f"<{a.ndim}I", *a.shape), a.tobytes()]
    return b"".join(out)


def unpack(data):
    if data[:4] != b"JWW1":
        raise ConversionError("el archivo no empieza con JWW1")
    (n,), off, p = struct.unpack("<I", data[4:8]), 8, {}
    for _ in range(n):
        (ln,), off = struct.unpack("<I", data[off:off + 4]), off + 4
        name, off = data[off:off + ln].decode(), off + ln
        (nd,), off = struct.unpack("<I", data[off:off + 4]), off + 4
        dims, off = struct.unpack(f"<{nd}I", data[off:off + 4 * nd]), off + 4 * nd
        count = int(np.prod(dims))
        p[name[3:]] = np.frombuffer(data[off:off + 4 * count], dtype="<f4").reshape(dims)
        off += 4 * count
    if off != len(data):
        raise ConversionError("sobran bytes al final del archivo")
    return p


def convert(onnx_path):
    import onnx
    import onnxruntime as ort

    model = onnx.load(onnx_path)
    consts = load_constants(model)
    p = by_name(consts) or by_graph(model, consts)
    if p is None:
        raise ConversionError("no encontré las 3 capas y las 2 LayerNorm del modelo 'dnn' de openWakeWord")
    p = check_shapes(p)

    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0]
    rng = np.random.default_rng(0)
    # Rasgos parecidos a los de verdad y también extremos.
    feats = np.concatenate([rng.normal(0, 3, (200, FEAT_FRAMES, EMB_DIM)),
                            rng.normal(0, 30, (40, FEAT_FRAMES, EMB_DIM)),
                            np.zeros((1, FEAT_FRAMES, EMB_DIM))]).astype(np.float32)
    ref = np.concatenate([np.asarray(sess.run(None, {inp.name: f[None]})[0]).reshape(-1) for f in feats])
    blob = pack(p)
    mine = forward(unpack(blob), feats)
    diff = float(np.max(np.abs(mine - ref)))
    if not np.isfinite(diff) or diff > 1e-4:
        raise ConversionError(f"la conversión no da lo mismo que el modelo original (diferencia {diff:.2e})")
    return blob, p["fc1.w"].shape[0], diff


def main(argv):
    if len(argv) != 3:
        print(__doc__)
        return 2
    try:
        blob, hidden, diff = convert(argv[1])
    except ConversionError as e:
        print(f"ERROR: {e}")
        return 1
    with open(argv[2], "wb") as f:
        f.write(blob)
    print(f"Listo: {argv[2]} ({len(blob)} bytes, capa oculta de {hidden} neuronas)")
    print(f"Revisado contra onnxruntime: diferencia máxima {diff:.1e}")
    print(f"SHA-256: {hashlib.sha256(blob).hexdigest()}")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "onnx_a_jww.py", "modelo/hey_sokari.onnx", "hey_sokari.jww"], capture_output=True, text=True)
print(r.stdout + r.stderr)
assert r.returncode == 0, "La conversión falló: manda este mensaje a Claude."
from google.colab import files
files.download("hey_sokari.jww")

## 7. Qué sigue

1. En GitHub abre el repo **sokari** → carpeta **`res`** → *Add file → Upload files*.
2. Arrastra **`hey_sokari.jww`**, elige *Create a new branch* y dale *Propose changes* → *Create pull request*.
3. El CI arma un `Sokari.exe` que ya trae "Hey Sokari". Bájalo de la pestaña *Actions* (artefacto **Sokari-exe**) y pruébalo unos días.

Si se activa solo con otras palabras o le cuesta oírte, se reentrena con más ejemplos o con otras frases en `custom_negative_phrases`.